# SeamlessM4T-v2 — Rigorous Validation

End-to-end validation of **SeamlessM4T-v2** inside the evaluator, across multiple settings:

1. **Audio synthesis** with M4T / MMS TTS — including the per-language (`language`) parameter.
2. **ASR backends** — Whisper vs SeamlessM4T ASR vs SeamlessM4T speech→English *translation*.
3. **Retrieval strategies** with M4T ASR — dense vs hybrid (RRF) vs reranker.
4. **Fusion retrieval** — `audio_text_retrieval` fusing an audio embedder with a text embedder.
5. **SONAR shared-space retrieval** — zero-shot audio→text in one 1024-d space.

Dataset: bundled `data/pubmed_qa_small/` (text questions → TTS audio → pipeline). Everything
runs on CPU with a small `TRACE_LIMIT`; bump it (and switch devices to CUDA) for a real run.

> SeamlessM4T-v2-large is a ~2.3 GB download; the first ASR/TTS cell will fetch it.

## 1. Install

In [1]:
# Install evaluator from repo root
%pip install -q -e '..'
# M4T ASR/TTS + audio IO
%pip install -q transformers torch sentencepiece soundfile

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Setup

Paths, a config builder, and a `run_setting` helper that runs one configuration and collects metrics.

In [2]:
from pathlib import Path
import pandas as pd
from evaluator import EvaluationConfig, run_evaluation

DATA_DIR = Path('data/pubmed_qa_small')
Q = str(DATA_DIR / 'questions.json')
C = str(DATA_DIR / 'corpus.json')
AUDIO_DIR = str(DATA_DIR / 'audio')
OUT_DIR = 'm4t_validation_out'

TRACE_LIMIT = 5          # raise for rigor (full set has 20 questions)
DEVICE = 'cpu'           # set 'cuda:0' if you have a GPU
EVAL_TTS = 'mms'         # fast TTS used to create ASR input audio

ROWS = []

def make_config(name, model, vector_db=None, features=None, trace_limit=TRACE_LIMIT):
    feats = {'audio_synthesis': {'enabled': True, 'provider': EVAL_TTS, 'language': 'en',
                                 'sample_rate': 16000, 'output_dir': AUDIO_DIR}}
    for k, v in (features or {}).items():
        feats[k] = {**feats.get(k, {}), **v} if isinstance(v, dict) else v
    return EvaluationConfig.from_dict({
        'experiment_name': name,
        'output_dir': OUT_DIR,
        'model': {'asr_device': DEVICE, 'text_emb_device': DEVICE, 'audio_emb_device': DEVICE, **model},
        'data': {'dataset_name': 'pubmed_qa', 'questions_path': Q, 'corpus_path': C,
                 'batch_size': 4, 'trace_limit': trace_limit, 'strict_validation': False},
        'vector_db': {'type': 'inmemory', 'k': 10, **(vector_db or {})},
        'features': feats,
        'cache': {'enabled': True},
    })

METRICS = ['WER', 'CER', 'MRR', 'MAP', 'Recall@1', 'Recall@5', 'NDCG@10']

def run_setting(label, model, mode='asr_text_retrieval', vector_db=None, features=None, trace_limit=TRACE_LIMIT):
    cfg = make_config(label, {'pipeline_mode': mode, **model}, vector_db, features, trace_limit)
    res = run_evaluation(cfg)
    row = {'setting': label}
    for k in METRICS:
        try:
            v = res.get_metric(k)
        except Exception:
            v = None
        if v is not None:
            row[k] = round(float(v), 4)
    ROWS.append(row)
    print(label, '->', {k: row[k] for k in row if k != 'setting'})
    return res

## 3. Audio synthesis + the `language` parameter

The evaluator synthesizes audio for text-only questions. `AudioSynthesisConfig.language` sets the
TTS language; each question's own `language` field (if present) overrides it, so multilingual
datasets synthesize in the right language. Here we synthesize the same idea in **English** and
**Polish** and listen to both.

In [3]:
from evaluator.config import AudioSynthesisConfig
from evaluator.pipeline.audio.synthesis import AudioSynthesizer
from IPython.display import Audio, display

samples = [
    ('en', 'Does aspirin reduce the risk of heart attack?'),
    ('pl', 'Czy aspiryna zmniejsza ryzyko zawalu serca?'),
]
for lang, text in samples:
    cfg = AudioSynthesisConfig(enabled=True, provider='mms', language=lang, sample_rate=16000)
    audio = AudioSynthesizer(cfg).synthesize(text)
    print(f'[{lang}] {text}')
    display(Audio(audio, rate=16000))

[en] Does aspirin reduce the risk of heart attack?


[pl] Czy aspiryna zmniejsza ryzyko zawalu serca?


In [4]:
# Optional: the same with the SeamlessM4T-v2 TTS provider (heavier; downloads the M4T model).
RUN_M4T_TTS = False
if RUN_M4T_TTS:
    cfg = AudioSynthesisConfig(enabled=True, provider='m4t', language='en', sample_rate=16000)
    audio = AudioSynthesizer(cfg).synthesize('Does aspirin reduce the risk of heart attack?')
    display(Audio(audio, rate=16000))

## 4. Experiment A — ASR backends

Same retrieval (LaBSE, dense), different ASR front-ends:

- `whisper` (tiny) — baseline
- `seamless_m4t` — SeamlessM4T-v2 ASR (English transcription)
- `seamless_m4t` + `tgt_lang=eng` — speech→English **translation** used as ASR

Lower WER and higher MRR are better. (English audio here, so ASR and translate behave similarly;
the translate setting shines on non-English audio.)

In [5]:
run_setting('whisper-tiny + labse', {'asr_model_type': 'whisper', 'asr_size': 'tiny', 'text_emb_model_type': 'labse'})
run_setting('m4t-asr + labse', {'asr_model_type': 'seamless_m4t', 'text_emb_model_type': 'labse'})
run_setting('m4t-translate(eng) + labse', {'asr_model_type': 'seamless_m4t', 'asr_params': {'tgt_lang': 'eng'}, 'text_emb_model_type': 'labse'})
pd.DataFrame(ROWS).set_index('setting')

INFO | evaluator | Logging initialized. Log file: logs/2026-06-03_20-59-54_whisper-tiny + labse.log
INFO | evaluator.services.model_services | service.start label=asr:whisper@cpu load_time=2.88s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.52s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 text question(s) via TTS provider 'mms' (lang=en) -> data/pubmed_qa_small/audio
INFO | evaluator.pipeline.audio.synthesis | TTS cache auto-enabled at: data/pubmed_qa_small/audio/.tts_cache
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthe

ASR Processing: 100%|██████████| 2/2 [00:12<00:00,  6.34s/it]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 12.68s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 2 batches processed
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================


INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)
INFO | evaluator.evaluation.phased | DB vectors stats - mean: -0.0070, std: 0.0354
INFO | evaluator.evaluation.phased | DB vectors norms - mean: 1.0000
INFO | evaluator.evaluation.phased | DB payload count: 20
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | RETRIEVAL DEBUG SAMPLE (first 3 queries):
INFO | evaluator.evaluation.

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

INFO | evaluator.services.model_services | service.start label=asr:seamless_m4t@cpu load_time=7.14s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.09s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 text question(s) via TTS provider 'mms' (lang=en) -> data/pubmed_qa_small/audio
INFO | evaluator.pipeline.audio.synthesis | TTS cache auto-enabled at: data/pubmed_qa_small/audio/.tts_cache
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.pipeline.audio.synthesis | TTS cache hit (fil

ASR Processing: 100%|██████████| 2/2 [00:10<00:00,  5.09s/it]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 10.18s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 2 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


,WER,CER,MRR,MAP,Recall@1,Recall@5,NDCG@10
setting,,,,,,,
whisper-tiny + labse,0.2853,0.0924,0.9,0.9,0.8,1.0,0.9262
m4t-asr + labse,0.3948,0.1075,0.9,0.9,0.8,1.0,0.9262
m4t-translate(eng) + labse,0.3948,0.1075,0.9,0.9,0.8,1.0,0.9262


## 5. Experiment B — retrieval strategies with M4T ASR

Fix the front-end to SeamlessM4T ASR + LaBSE; vary retrieval:

- **dense** — embedding similarity
- **hybrid (RRF)** — dense + BM25 fused with reciprocal-rank fusion
- **dense + reranker** — cross-encoder reranks top candidates (downloads a small cross-encoder)

In [6]:
M4T = {'asr_model_type': 'seamless_m4t', 'text_emb_model_type': 'labse'}

run_setting('m4t + dense', M4T, vector_db={'retrieval_mode': 'dense'})
run_setting('m4t + hybrid(rrf)', M4T, vector_db={'retrieval_mode': 'hybrid', 'hybrid_fusion_method': 'rrf'})

RUN_RERANKER = True  # set False to skip the cross-encoder download
if RUN_RERANKER:
    run_setting('m4t + dense + reranker', M4T, vector_db={
        'retrieval_mode': 'dense', 'reranker_enabled': True, 'reranker_mode': 'cross_encoder',
        'reranker_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2', 'reranker_top_k': 10,
    })
pd.DataFrame(ROWS).set_index('setting')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

INFO | evaluator.services.model_services | service.start label=asr:seamless_m4t@cpu load_time=6.97s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.07s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 text question(s) via TTS provider 'mms' (lang=en) -> data/pubmed_qa_small/audio
INFO | evaluator.pipeline.audio.synthesis | TTS cache auto-enabled at: data/pubmed_qa_small/audio/.tts_cache
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.pipeline.audio.synthesis | TTS cache hit (fil

ASR Processing: 100%|██████████| 2/2 [00:09<00:00,  4.87s/it]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 9.75s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 2 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


...

INFO | evaluator.services.model_services | service.start label=asr:seamless_m4t@cpu load_time=7.22s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.17s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore, cross-encoder reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 text question(s) via TTS provider 'mms' (lang=en) -> data/pubmed_qa_small/audio
INFO | evaluator.pipeline.audio.synthesis | TTS cache auto-enabled at: data/pubmed_qa_small/audio/.tts_cache
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms


ASR Processing: 100%|██████████| 2/2 [00:09<00:00,  4.98s/it]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 9.95s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 2 batches processed
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions


INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.61s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)
INFO | evaluator.evaluation.phased | DB vectors stats - mean: -0.0070, std: 0.0354
INFO | evaluator.evaluation.phased | DB vectors norms - mean: 1.0000
INFO 

,WER,CER,MRR,MAP,Recall@1,Recall@5,NDCG@10
setting,,,,,,,
whisper-tiny + labse,0.2853,0.0924,0.90,0.90,0.8,1.0,0.9262
m4t-asr + labse,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t-translate(eng) + labse,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t + dense,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t + hybrid(rrf),0.3948,0.1075,0.84,0.84,0.8,1.0,0.8774
m4t + dense + reranker,0.3948,0.1075,1.00,1.00,1.0,1.0,1.0000


## 6. Experiment C — fusion retrieval (`audio_text_retrieval`)

Cross-modal fusion: embed the **audio** query (HuBERT, pretrained) and the **text** corpus
(LaBSE), then fuse query representations with `embedding_fusion`. We sweep `audio_weight`.

> HuBERT is used here because it needs no training. For an M4T-native audio embedder use
> `attention_pool_m4t` (needs a trained projection head) or the SONAR speech encoder
> (`sonar_speech`) shown in the next section.

In [7]:
for aw in (0.3, 0.5, 0.7):
    run_setting(
        f'fusion hubert+labse (audio_w={aw})',
        {'audio_emb_model_type': 'hubert', 'audio_emb_dim': 768, 'text_emb_model_type': 'labse'},
        mode='audio_text_retrieval',
        features={'embedding_fusion': {'enabled': True, 'fusion_method': 'weighted',
                                       'audio_weight': aw, 'text_weight': round(1 - aw, 2)}},
    )
pd.DataFrame(ROWS).set_index('setting')

INFO | evaluator.services.model_services | service.start label=audio_embedding:hubert@cpu load_time=1.05s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.09s
INFO | evaluator.pipeline.audio_embedding_pipeline | AudioEmbedding pipeline initialized with model: HuBERTAudioModel - model:facebook/hubert-base-ls960 - pooling:mean - dim:768
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 text question(s) via TTS provider 'mms' (lang=en) -> data/pubmed_qa_small/audio
INFO | evaluator.pipeline.audio.synthesis | TTS cache auto-enabled at: data/pubmed_qa_small/audio/.tts_cache
INFO | evaluator.models.tts.mms_tts | MM

Audio embedding:   0%|          | 0/2 [00:00<?, ?it/s]

INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 4/4 hits
INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 1/1 hits


Audio embedding: 100%|██████████| 2/2 [00:00<00:00, 79.27it/s]

INFO | evaluator.evaluation.phased | Audio Embedding Phase completed in 0.03s
INFO | evaluator.evaluation.phased | Audio Embedding Phase complete: 5 audio embeddings
INFO | evaluator.evaluation.phased | Audio embedding shape: (5, 768)
INFO | evaluator.evaluation.phased | Audio embedding stats - mean: 0.0013, std: 0.0361
INFO | evaluator.evaluation.phased | Audio embedding norms - mean: 1.0000
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 1.5: Text Embedding & Fusion
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Text Embedding for Fusion completed in 0.02s
INFO | evaluator.evaluation.phased | Text embedding shape: (5, 768)
INFO | evaluator.evaluation.phased | Text embedding stats - mean: -0.0106, std: 0.0345
INFO | evaluator.models.retrieval.embedding_fusion | Fusion config validated: method=weighted, audio_dim=768, text_dim=

...

INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 4/4 hits
INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 1/1 hits


Audio embedding: 100%|██████████| 2/2 [00:00<00:00, 78.02it/s]

INFO | evaluator.evaluation.phased | Audio Embedding Phase completed in 0.03s
INFO | evaluator.evaluation.phased | Audio Embedding Phase complete: 5 audio embeddings
INFO | evaluator.evaluation.phased | Audio embedding shape: (5, 768)
INFO | evaluator.evaluation.phased | Audio embedding stats - mean: 0.0013, std: 0.0361
INFO | evaluator.evaluation.phased | Audio embedding norms - mean: 1.0000
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 1.5: Text Embedding & Fusion
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Text Embedding for Fusion completed in 0.02s
INFO | evaluator.evaluation.phased | Text embedding shape: (5, 768)
INFO | evaluator.evaluation.phased | Text embedding stats - mean: -0.0106, std: 0.0345
INFO | evaluator.models.retrieval.embedding_fusion | Fusion config validated: method=weighted, audio_dim=768, text_dim=

INFO | evaluator.services.model_services | service.stop label=text_embedding:labse@cpu
INFO | evaluator.services.model_services | service.stop label=audio_embedding:hubert@cpu
fusion hubert+labse (audio_w=0.7) -> {'MRR': 0.9, 'MAP': 0.9, 'Recall@1': 0.8, 'Recall@5': 1.0, 'NDCG@10': 0.9262}


,WER,CER,MRR,MAP,Recall@1,Recall@5,NDCG@10
setting,,,,,,,
whisper-tiny + labse,0.2853,0.0924,0.90,0.90,0.8,1.0,0.9262
m4t-asr + labse,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t-translate(eng) + labse,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t + dense,0.3948,0.1075,0.90,0.90,0.8,1.0,0.9262
m4t + hybrid(rrf),0.3948,0.1075,0.84,0.84,0.8,1.0,0.8774
m4t + dense + reranker,0.3948,0.1075,1.00,1.00,1.0,1.0,1.0000
fusion hubert+labse (audio_w=0.3),NaN,NaN,1.00,1.00,1.0,1.0,1.0000
fusion hubert+labse (audio_w=0.5),NaN,NaN,1.00,1.00,1.0,1.0,1.0000
fusion hubert+labse (audio_w=0.7),NaN,NaN,0.90,0.90,0.8,1.0,0.9262


## 7. Experiment D — SONAR shared-space retrieval

SONAR's speech and text encoders (the embedding side of the Seamless/M4T family) share one
**1024-d space**, so an **audio query** can retrieve a **text corpus** directly — zero-shot,
no trained head and no fusion. This is the M4T/SONAR cross-modal setting
(`audio_text_retrieval` with `sonar_speech` query encoder + `sonar` text encoder).

> Requires `pip install -e '..[sonar]'` (pulls sonar-space + fairseq2). Gated by `RUN_SONAR`.

In [4]:
RUN_SONAR = True  # set True after: pip install -e '..[sonar]'
if RUN_SONAR:
    run_setting(
        'sonar cross-modal (audio->text)',
        {'asr_model_type': None,
         'audio_emb_model_type': 'sonar_speech', 'audio_emb_model_name': 'sonar_speech_encoder_eng',
         'audio_emb_dim': 1024, 'text_emb_model_type': 'sonar'},
        mode='audio_text_retrieval',
        features={'embedding_fusion': {'enabled': False}},  # shared space -> compare directly
    )
    display(pd.DataFrame(ROWS).set_index('setting'))
else:
    print('RUN_SONAR=False - install evaluator[sonar] and set RUN_SONAR=True to run SONAR cross-modal retrieval.')

INFO | evaluator | Logging initialized. Log file: logs/2026-06-03_22-48-50_sonar cross-modal (audio->text).log
INFO | evaluator.services.model_provider | provider.shutdown offload=True


EvaluationError: Failed to create pipelines: SONAR not installed. Install with: pip install 'evaluator[sonar]'
(pulls sonar-space + fairseq2).

In [5]:
!pip freeze | grep fairseq
!pip freeze | grep sonar

fairseq2==0.8.1
fairseq2n==0.8.1
sonar-space==0.5.0


## 8. Summary

All settings side by side, with an MRR bar chart.

In [6]:
import matplotlib.pyplot as plt

df = pd.DataFrame(ROWS).set_index('setting')
display(df)

if 'MRR' in df.columns:
    ax = df['MRR'].plot(kind='barh', figsize=(8, 0.5 * len(df) + 1), color='#1d4ed8')
    ax.set_xlabel('MRR'); ax.set_title('Retrieval quality (MRR) by setting')
    plt.tight_layout(); plt.show()

KeyError: "None of ['setting'] are in the columns"

## 9. Scaling up

- **More data / rigor:** raise `TRACE_LIMIT` (full bundled set = 20) or point `Q`/`C` at
  `data/pubmedqa/` (larger) or your own `questions.json` / `corpus.json`.
- **GPU:** set `DEVICE = 'cuda:0'` for much faster M4T.
- **Non-English audio:** set per-question `language` in the dataset (or `audio_synthesis.language`)
  and use `m4t-translate(eng)` to retrieve foreign-language speech against an English corpus.
- **Matrix runs:** `from evaluator import run_evaluation_matrix` to sweep many setups over one
  base config in a single call.
- **SONAR cross-modal:** `pip install -e '..[sonar]'` then set `RUN_SONAR = True` above for
  zero-shot audio→text retrieval in the shared SONAR space.